# 06·04 — RAG Evaluation: Measuring Retrieval + Generation Quality

> **Module:** 06 – LLM Evaluation  
> **Notebook:** 4 of 4  
> **Objective:** Build a comprehensive RAG evaluation framework that measures every component of the pipeline independently and holistically.

---

## 🎯 Learning Objectives

1. Understand why RAG evaluation requires component-level decomposition
2. Implement Retrieval metrics: Precision@K, Recall@K, MRR, NDCG
3. Implement Generation metrics: faithfulness, answer relevance, context utilization
4. Build the RAGAS framework metrics from scratch
5. Diagnose whether failures come from retrieval or generation

---

## 1. Why RAG Evaluation is Uniquely Hard

RAG has TWO subsystems that can fail independently:

```
┌─────────────────────────────────────────────────────────────────────┐
│               RAG FAILURE MODE MATRIX                                │
│                                                                       │
│              Retrieval GOOD     Retrieval BAD                        │
│  ┌────────────────────────────────────────────────────┐              │
│  │ Gen GOOD │  ✅ IDEAL         │ 🔴 Lucky hallucination│            │
│  │          │  Best case        │ (got right answer     │            │
│  │          │                   │  despite bad context) │            │
│  ├──────────┼───────────────────┼─────────────────────-┤            │
│  │ Gen BAD  │ 🟡 Missed answer  │ 🔴 Complete failure  │            │
│  │          │ (right docs,      │                       │            │
│  │          │  wrong synthesis) │                       │            │
│  └────────────────────────────────────────────────────┘              │
│                                                                       │
│  Evaluation must identify WHICH component failed.                    │
│  End-to-end accuracy alone cannot diagnose this.                     │
└─────────────────────────────────────────────────────────────────────┘
```

### The Three Evaluation Dimensions

```
1. RETRIEVAL QUALITY
   - Did we retrieve the relevant documents?
   - How high were they ranked?
   Metrics: Precision@K, Recall@K, MRR, NDCG

2. GENERATION FAITHFULNESS  
   - Does the answer contain only claims supported by the retrieved context?
   - Are there hallucinated facts not grounded in the context?
   Metrics: Faithfulness score, citation accuracy

3. ANSWER RELEVANCE
   - Does the answer actually address the original question?
   - Is it complete and useful?
   Metrics: Answer relevance, completeness, helpfulness
```

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
!pip install sentence-transformers faiss-cpu numpy pandas matplotlib seaborn -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Set, Tuple
from sentence_transformers import SentenceTransformer
import math
import json
import warnings
warnings.filterwarnings('ignore')

print("Setup complete.")

## 2. Retrieval Metrics: The Full Toolkit

In [ ]:
@dataclass
class RetrievalExample:
    query: str
    relevant_doc_ids: Set[str]     # ground truth
    retrieved_doc_ids: List[str]   # ordered by rank (best first)


class RetrievalMetrics:
    """
    Complete retrieval evaluation toolkit.

    All metrics assume retrieved_doc_ids is ordered by relevance (rank 1 = best).
    """

    @staticmethod
    def precision_at_k(example: RetrievalExample, k: int) -> float:
        """
        Fraction of top-K retrieved docs that are relevant.

        P@K = |{relevant} ∩ {top-K retrieved}| / K

        Answers: "Of the K docs I retrieved, how many are actually useful?"
        Use when: retrieval slot is limited (short prompts, cost-sensitive)
        """
        top_k = example.retrieved_doc_ids[:k]
        hits = sum(1 for doc_id in top_k if doc_id in example.relevant_doc_ids)
        return hits / k

    @staticmethod
    def recall_at_k(example: RetrievalExample, k: int) -> float:
        """
        Fraction of all relevant docs found in top-K.

        R@K = |{relevant} ∩ {top-K retrieved}| / |relevant|

        Answers: "Of all the relevant docs that exist, how many did I find?"
        Use when: missing a relevant doc is costly (research, compliance)
        """
        if not example.relevant_doc_ids:
            return 1.0
        top_k = set(example.retrieved_doc_ids[:k])
        hits = len(example.relevant_doc_ids & top_k)
        return hits / len(example.relevant_doc_ids)

    @staticmethod
    def mrr(example: RetrievalExample) -> float:
        """
        Mean Reciprocal Rank — rewards systems that rank the first relevant doc high.

        RR = 1/rank_of_first_relevant_doc
        MRR = mean of RR over all queries

        Answers: "How far do I have to scroll to find the first useful result?"
        Use when: user looks at results sequentially and stops at first relevant one
        """
        for rank, doc_id in enumerate(example.retrieved_doc_ids, start=1):
            if doc_id in example.relevant_doc_ids:
                return 1.0 / rank
        return 0.0

    @staticmethod
    def average_precision(example: RetrievalExample) -> float:
        """
        Area under the Precision-Recall curve — rewards consistent ranking of relevant docs.

        AP = (1/|relevant|) · Σ P@k · rel(k)

        Answers: "How good is the ranking of ALL relevant docs?"
        Use when: you care about the entire ranked list, not just top-K
        """
        if not example.relevant_doc_ids:
            return 1.0
        hits, running_precision = 0, 0.0
        for rank, doc_id in enumerate(example.retrieved_doc_ids, start=1):
            if doc_id in example.relevant_doc_ids:
                hits += 1
                running_precision += hits / rank
        return running_precision / len(example.relevant_doc_ids)

    @staticmethod
    def ndcg_at_k(example: RetrievalExample, k: int,
                  relevance_grades: Optional[Dict[str, int]] = None) -> float:
        """
        Normalized Discounted Cumulative Gain — accounts for graded relevance and position.

        DCG@K = Σ (2^rel_i - 1) / log2(i + 1)   for i = 1..K
        NDCG@K = DCG@K / IDCG@K                   (normalized by ideal ranking)

        Answers: "How well does the ranking match the ideal relevance ordering?"
        Use when: documents have different relevance levels (binary is a special case)
        """
        def dcg(doc_ids: List[str], k: int) -> float:
            score = 0.0
            for i, doc_id in enumerate(doc_ids[:k], start=1):
                rel = relevance_grades.get(doc_id, 0) if relevance_grades else (
                    1 if doc_id in example.relevant_doc_ids else 0
                )
                score += (2**rel - 1) / math.log2(i + 1)
            return score

        actual_dcg = dcg(example.retrieved_doc_ids, k)

        # Ideal DCG: sort all relevant docs by relevance descending
        if relevance_grades:
            ideal_order = sorted(relevance_grades.keys(),
                                 key=lambda x: relevance_grades[x], reverse=True)
        else:
            ideal_order = list(example.relevant_doc_ids)

        ideal_dcg = dcg(ideal_order, k)
        return actual_dcg / ideal_dcg if ideal_dcg > 0 else 0.0

    @classmethod
    def evaluate_all(
        cls,
        examples: List[RetrievalExample],
        k_values: List[int] = [1, 3, 5, 10],
    ) -> pd.DataFrame:
        """Compute all metrics across all examples and k values."""
        records = []
        m = cls()
        for ex in examples:
            row = {'query': ex.query[:50], 'mrr': m.mrr(ex), 'map': m.average_precision(ex)}
            for k in k_values:
                row[f'P@{k}'] = m.precision_at_k(ex, k)
                row[f'R@{k}'] = m.recall_at_k(ex, k)
                row[f'NDCG@{k}'] = m.ndcg_at_k(ex, k)
            records.append(row)
        return pd.DataFrame(records)


# ── Demo: Compare two retrieval systems ───────────────────────────────────
examples_system_A = [
    RetrievalExample(
        query="How does attention work in transformers?",
        relevant_doc_ids={"doc_attention", "doc_transformer", "doc_softmax"},
        retrieved_doc_ids=["doc_attention", "doc_transformer", "doc_rnn", "doc_cnn", "doc_softmax"],
    ),
    RetrievalExample(
        query="What is gradient descent?",
        relevant_doc_ids={"doc_optim", "doc_backprop"},
        retrieved_doc_ids=["doc_optim", "doc_backprop", "doc_activations", "doc_loss", "doc_batch"],
    ),
    RetrievalExample(
        query="Explain the KV cache mechanism",
        relevant_doc_ids={"doc_kvcache", "doc_inference"},
        retrieved_doc_ids=["doc_inference", "doc_tokenizer", "doc_sampling", "doc_kvcache", "doc_batching"],
    ),
]

examples_system_B = [
    RetrievalExample(
        query="How does attention work in transformers?",
        relevant_doc_ids={"doc_attention", "doc_transformer", "doc_softmax"},
        retrieved_doc_ids=["doc_cnn", "doc_rnn", "doc_attention", "doc_softmax", "doc_transformer"],
    ),
    RetrievalExample(
        query="What is gradient descent?",
        relevant_doc_ids={"doc_optim", "doc_backprop"},
        retrieved_doc_ids=["doc_loss", "doc_optim", "doc_activations", "doc_batch", "doc_backprop"],
    ),
    RetrievalExample(
        query="Explain the KV cache mechanism",
        relevant_doc_ids={"doc_kvcache", "doc_inference"},
        retrieved_doc_ids=["doc_batching", "doc_tokenizer", "doc_sampling", "doc_inference", "doc_kvcache"],
    ),
]

metrics = RetrievalMetrics()
df_A = metrics.evaluate_all(examples_system_A)
df_B = metrics.evaluate_all(examples_system_B)

print("System A (relevant docs ranked higher):")
print(df_A[['query', 'P@3', 'R@5', 'mrr', 'NDCG@5']].to_string(index=False))
print(f"\nMEAN:  P@3={df_A['P@3'].mean():.3f}  R@5={df_A['R@5'].mean():.3f}  "
      f"MRR={df_A['mrr'].mean():.3f}  NDCG@5={df_A['NDCG@5'].mean():.3f}")

print("\nSystem B (relevant docs ranked lower):")
print(df_B[['query', 'P@3', 'R@5', 'mrr', 'NDCG@5']].to_string(index=False))
print(f"\nMEAN:  P@3={df_B['P@3'].mean():.3f}  R@5={df_B['R@5'].mean():.3f}  "
      f"MRR={df_B['mrr'].mean():.3f}  NDCG@5={df_B['NDCG@5'].mean():.3f}")

In [ ]:
# Precision-Recall curve across K values
k_values = [1, 2, 3, 4, 5]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# P-R Curve
ax = axes[0]
for name, examples in [("System A", examples_system_A), ("System B", examples_system_B)]:
    m = RetrievalMetrics()
    precs = [np.mean([m.precision_at_k(e, k) for e in examples]) for k in k_values]
    recs  = [np.mean([m.recall_at_k(e, k) for e in examples]) for k in k_values]
    ax.plot(recs, precs, 'o-', label=name, linewidth=2, markersize=8)
    for k, r, p in zip(k_values, recs, precs):
        ax.annotate(f'K={k}', (r, p), textcoords='offset points', xytext=(5, 3), fontsize=8)
ax.set_xlabel('Recall@K', fontsize=12)
ax.set_ylabel('Precision@K', fontsize=12)
ax.set_title('Precision-Recall Curve by K', fontsize=12)
ax.legend()
ax.grid(alpha=0.3)
ax.set_xlim(-0.05, 1.1)
ax.set_ylim(-0.05, 1.1)

# MRR comparison
ax = axes[1]
m = RetrievalMetrics()
mrr_A = [m.mrr(e) for e in examples_system_A]
mrr_B = [m.mrr(e) for e in examples_system_B]
queries = [e.query[:25] for e in examples_system_A]
x = np.arange(len(queries))
ax.bar(x - 0.2, mrr_A, 0.35, label='System A', alpha=0.85)
ax.bar(x + 0.2, mrr_B, 0.35, label='System B', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(queries, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('MRR (higher = better)', fontsize=11)
ax.set_title('Mean Reciprocal Rank by Query', fontsize=12)
ax.legend()
ax.grid(alpha=0.3, axis='y')
ax.set_ylim(0, 1.15)

# NDCG across K
ax = axes[2]
for name, examples in [("System A", examples_system_A), ("System B", examples_system_B)]:
    m = RetrievalMetrics()
    ndcgs = [np.mean([m.ndcg_at_k(e, k) for e in examples]) for k in k_values]
    ax.plot(k_values, ndcgs, 'o-', label=name, linewidth=2, markersize=8)
ax.set_xlabel('K', fontsize=12)
ax.set_ylabel('NDCG@K', fontsize=12)
ax.set_title('NDCG as K Increases', fontsize=12)
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(0, 1.1)

plt.suptitle('Retrieval Evaluation Metrics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/06_retrieval_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Generation Metrics: Faithfulness & Relevance

### The RAGAS Framework

RAGAS (Retrieval Augmented Generation Assessment) decomposes RAG quality into four orthogonal dimensions:

```
1. FAITHFULNESS
   Are ALL claims in the answer supported by the retrieved context?
   Score = supported_claims / total_claims_in_answer
   Detects: hallucination, fabrication

2. ANSWER RELEVANCE
   Does the answer address the original question?
   Score = cosine_similarity(answer_embedding, question_embedding)
   Detects: off-topic answers, tangential responses

3. CONTEXT PRECISION
   Are the retrieved contexts actually relevant to the question?
   Score = relevant_contexts / retrieved_contexts
   Detects: noisy retrieval returning irrelevant chunks

4. CONTEXT RECALL
   Does the retrieved context contain all the information needed?
   Score = attributable_reference_sentences / total_reference_sentences
   Detects: missing information in retrieved context
```

In [ ]:
# Load a small embedding model for semantic similarity
print("Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("Ready.")


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))


class RAGMetrics:
    """
    RAGAS-inspired metrics for evaluating RAG pipeline quality.
    Implements faithfulness, answer relevance, and context precision.
    """

    def __init__(self, embed_model=None):
        self.embedder = embed_model or embedder

    def faithfulness(
        self,
        answer: str,
        contexts: List[str],
        llm_judge=None,
    ) -> Dict:
        """
        Measure whether every claim in the answer is supported by the retrieved contexts.

        Full RAGAS uses an LLM to extract claims and verify them.
        This implementation uses semantic similarity as a proxy.

        Faithfulness = supported_sentences / total_sentences
        """
        import re
        # Split answer into individual sentences (claim units)
        sentences = [s.strip() for s in re.split(r'[.!?]+', answer) if len(s.strip()) > 20]

        if not sentences:
            return {'faithfulness': 1.0, 'n_sentences': 0, 'details': []}

        # Embed all answer sentences and context
        sentence_embs = self.embedder.encode(sentences, normalize_embeddings=True)
        context_text = ' '.join(contexts)
        context_chunks = [c[:200] for c in contexts]  # chunk contexts
        context_embs = self.embedder.encode(context_chunks, normalize_embeddings=True)

        FAITHFULNESS_THRESHOLD = 0.45  # tune based on your embedding model

        details = []
        supported_count = 0
        for sent, sent_emb in zip(sentences, sentence_embs):
            max_sim = max(
                cosine_similarity(sent_emb, ctx_emb)
                for ctx_emb in context_embs
            ) if len(context_embs) > 0 else 0.0
            is_supported = max_sim >= FAITHFULNESS_THRESHOLD
            supported_count += int(is_supported)
            details.append({
                'sentence': sent[:80],
                'max_similarity': round(max_sim, 3),
                'supported': is_supported
            })

        return {
            'faithfulness': round(supported_count / len(sentences), 3),
            'n_sentences': len(sentences),
            'n_supported': supported_count,
            'details': details,
        }

    def answer_relevance(
        self,
        question: str,
        answer: str,
        n_synthetic_questions: int = 3,
    ) -> Dict:
        """
        Measure if the answer addresses the question.

        RAGAS approach: generate synthetic questions from the answer,
        then measure cosine similarity between synthetic Qs and the original Q.
        Here we use a proxy: direct cosine similarity + coverage check.
        """
        q_emb = self.embedder.encode([question], normalize_embeddings=True)[0]
        a_emb = self.embedder.encode([answer], normalize_embeddings=True)[0]

        # Direct semantic similarity between question and answer
        direct_sim = cosine_similarity(q_emb, a_emb)

        # Keyword overlap as additional signal
        import re
        q_words = set(re.findall(r'\b\w{4,}\b', question.lower()))
        a_words = set(re.findall(r'\b\w{4,}\b', answer.lower()))
        keyword_overlap = len(q_words & a_words) / max(len(q_words), 1)

        relevance = 0.7 * direct_sim + 0.3 * keyword_overlap

        return {
            'answer_relevance': round(float(relevance), 3),
            'semantic_similarity': round(float(direct_sim), 3),
            'keyword_overlap': round(keyword_overlap, 3),
        }

    def context_precision(
        self,
        question: str,
        contexts: List[str],
        relevance_threshold: float = 0.35,
    ) -> Dict:
        """
        Fraction of retrieved contexts that are relevant to the question.

        Low context precision = noisy retrieval = model must ignore irrelevant chunks.
        This wastes context window and can confuse the model.
        """
        q_emb = self.embedder.encode([question], normalize_embeddings=True)[0]
        ctx_embs = self.embedder.encode(contexts, normalize_embeddings=True)

        relevances = [cosine_similarity(q_emb, c_emb) for c_emb in ctx_embs]
        n_relevant = sum(1 for r in relevances if r >= relevance_threshold)

        return {
            'context_precision': round(n_relevant / max(len(contexts), 1), 3),
            'n_contexts': len(contexts),
            'n_relevant': n_relevant,
            'relevance_scores': [round(r, 3) for r in relevances],
        }

    def evaluate_full(
        self,
        question: str,
        answer: str,
        contexts: List[str],
    ) -> Dict:
        """Run all four RAGAS metrics and compute an aggregate score."""
        faith = self.faithfulness(answer, contexts)
        rel   = self.answer_relevance(question, answer)
        prec  = self.context_precision(question, contexts)

        # Aggregate: harmonic mean of core metrics
        scores = [faith['faithfulness'], rel['answer_relevance'], prec['context_precision']]
        ragas_score = len(scores) / sum(1.0/(s + 1e-9) for s in scores)

        return {
            'ragas_score': round(ragas_score, 3),
            'faithfulness': faith['faithfulness'],
            'answer_relevance': rel['answer_relevance'],
            'context_precision': prec['context_precision'],
            'faithfulness_details': faith['details'],
            'context_relevances': prec['relevance_scores'],
        }


rag_metrics = RAGMetrics()
print("RAGMetrics initialized.")

In [ ]:
# ── Evaluate different RAG failure modes ──────────────────────────────────

question = "What are the main advantages of the transformer architecture over RNNs?"

good_contexts = [
    "Transformers process all tokens in parallel using self-attention, unlike RNNs which process sequentially.",
    "The attention mechanism allows transformers to capture long-range dependencies more effectively than RNNs.",
    "Transformer training is significantly faster due to parallelization, addressing the RNN bottleneck.",
]

bad_contexts = [
    "The history of artificial intelligence dates back to the 1950s with Alan Turing's seminal work.",
    "Python is a popular programming language for machine learning due to its simplicity and libraries.",
    "Recurrent neural networks use backpropagation through time for training.",  # slightly relevant
]

test_cases = [
    {
        "name": "✅ IDEAL: Faithful, relevant, good context",
        "answer": "Transformers offer three key advantages over RNNs. First, parallelization: transformers process all tokens simultaneously using self-attention, making training much faster. Second, long-range dependencies: attention can directly connect any two positions, regardless of distance. Third, no vanishing gradient problem that plagues RNNs with long sequences.",
        "contexts": good_contexts,
    },
    {
        "name": "🔴 HALLUCINATION: Claims not in context",
        "answer": "Transformers are better because they use GPU clusters with 8192 cores and require exactly 3 terabytes of training data. The transformer was invented by Yann LeCun in 2015 at Facebook AI Research. It achieves 99.7% accuracy on all NLP tasks.",
        "contexts": good_contexts,
    },
    {
        "name": "🟡 OFF-TOPIC: Irrelevant answer",
        "answer": "Python is a great programming language for machine learning. You should install PyTorch and TensorFlow. The history of AI is fascinating and dates back to the 1950s.",
        "contexts": good_contexts,
    },
    {
        "name": "🔴 BAD RETRIEVAL: Noisy context, ok answer",
        "answer": "Transformers process tokens in parallel, unlike RNNs which process them sequentially. This enables faster training and better capture of long-range dependencies.",
        "contexts": bad_contexts,
    },
    {
        "name": "🟡 INCOMPLETE: Partial answer",
        "answer": "Transformers are faster than RNNs.",
        "contexts": good_contexts,
    },
]

print(f"Question: {question}")
print()
print(f"{'Case':<45} {'RAGAS':>6} {'Faith':>6} {'Rel':>6} {'Prec':>6}")
print("-" * 73)

all_results = []
for tc in test_cases:
    r = rag_metrics.evaluate_full(question, tc['answer'], tc['contexts'])
    all_results.append((tc['name'], r))
    print(f"{tc['name']:<45} {r['ragas_score']:>6.3f} {r['faithfulness']:>6.3f} "
          f"{r['answer_relevance']:>6.3f} {r['context_precision']:>6.3f}")

In [ ]:
# Radar chart showing per-metric scores per case
from matplotlib.patches import FancyArrowPatch

fig, axes = plt.subplots(1, 5, figsize=(20, 5), subplot_kw=dict(polar=True))
metric_names = ['Faithfulness', 'Answer\nRelevance', 'Context\nPrecision']
n_metrics = len(metric_names)
angles = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

colors = ['#2ca02c', '#d62728', '#ff7f0e', '#d62728', '#ff7f0e']

for ax, (case_name, r), color in zip(axes, all_results, colors):
    values = [r['faithfulness'], r['answer_relevance'], r['context_precision']]
    values += values[:1]  # close polygon

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metric_names, fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['', '', '', ''], fontsize=7)

    ax.plot(angles, values, color=color, linewidth=2)
    ax.fill(angles, values, color=color, alpha=0.25)
    ax.set_title(f'{case_name[:25]}\nRAGAS={r["ragas_score"]:.2f}',
                 size=9, fontweight='bold', pad=12)

plt.suptitle('RAGAS Metric Breakdown by Case', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/06_ragas_radar.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. End-to-End RAG Benchmark: Diagnosing the Pipeline

In [ ]:
class RAGBenchmark:
    """
    Complete RAG benchmark that runs a test set and produces a diagnostic report.
    
    The key insight: always measure retrieval and generation SEPARATELY.
    End-to-end accuracy alone cannot tell you WHERE the pipeline is failing.
    """

    def __init__(
        self,
        retrieval_system,   # must have .search(query, k) -> List[str]
        generation_fn,      # must have .generate(question, contexts) -> str
        rag_metrics: RAGMetrics,
        retrieval_metrics: RetrievalMetrics,
    ):
        self.retrieval = retrieval_system
        self.generator = generation_fn
        self.rag_metrics = rag_metrics
        self.ret_metrics = retrieval_metrics

    def run(
        self,
        test_cases: List[Dict],
        k: int = 5,
    ) -> pd.DataFrame:
        """
        Run the full benchmark.

        test_cases format:
        [
            {
                "question": str,
                "reference_answer": str,
                "relevant_doc_ids": set   # for retrieval eval
            }
        ]
        """
        records = []
        for tc in test_cases:
            # 1. Retrieve
            retrieved = self.retrieval.search(tc['question'], k=k)
            retrieved_ids = [r.doc_id for r in retrieved]
            contexts = [r.text for r in retrieved]

            # 2. Generate
            answer = self.generator.generate(tc['question'], contexts)

            # 3. Evaluate retrieval
            ret_ex = RetrievalExample(
                query=tc['question'],
                relevant_doc_ids=set(tc.get('relevant_doc_ids', [])),
                retrieved_doc_ids=retrieved_ids
            )

            # 4. Evaluate generation
            gen_scores = self.rag_metrics.evaluate_full(tc['question'], answer, contexts)

            records.append({
                'question': tc['question'][:50],
                'P@3':   self.ret_metrics.precision_at_k(ret_ex, 3),
                'R@5':   self.ret_metrics.recall_at_k(ret_ex, 5),
                'MRR':   self.ret_metrics.mrr(ret_ex),
                **{k: v for k, v in gen_scores.items()
                   if k not in ('faithfulness_details', 'context_relevances')},
                'answer': answer[:100],
            })

        return pd.DataFrame(records)

    def diagnose(self, results: pd.DataFrame) -> str:
        """Produce a human-readable diagnostic report."""
        mean = results.mean(numeric_only=True)
        report = ["\n" + "═"*60, "RAG PIPELINE DIAGNOSTIC REPORT", "═"*60]

        # Retrieval diagnosis
        report.append("\n📥 RETRIEVAL QUALITY")
        report.append(f"  Precision@3: {mean.get('P@3', 0):.3f}")
        report.append(f"  Recall@5:    {mean.get('R@5', 0):.3f}")
        report.append(f"  MRR:         {mean.get('MRR', 0):.3f}")

        retrieval_ok = mean.get('P@3', 0) >= 0.5 and mean.get('R@5', 0) >= 0.6
        report.append(f"  Status: {'✅ GOOD' if retrieval_ok else '❌ NEEDS IMPROVEMENT'}")
        if not retrieval_ok:
            report.append("  → Try: better embeddings, hybrid search, query rewriting")

        # Generation diagnosis
        report.append("\n📤 GENERATION QUALITY")
        report.append(f"  Faithfulness:     {mean.get('faithfulness', 0):.3f}")
        report.append(f"  Answer Relevance: {mean.get('answer_relevance', 0):.3f}")
        report.append(f"  Context Precision:{mean.get('context_precision', 0):.3f}")
        report.append(f"  RAGAS Score:      {mean.get('ragas_score', 0):.3f}")

        faith_ok = mean.get('faithfulness', 0) >= 0.7
        rel_ok   = mean.get('answer_relevance', 0) >= 0.6
        report.append(f"  Faithfulness: {'✅ GOOD' if faith_ok else '❌ HALLUCINATING'}")
        report.append(f"  Relevance:    {'✅ GOOD' if rel_ok else '❌ OFF-TOPIC'}")
        if not faith_ok:
            report.append("  → Try: stronger faithfulness prompt, post-generation fact checking")
        if not rel_ok:
            report.append("  → Try: better system prompt, query rewriting")

        report.append("\n" + "═"*60)
        return "\n".join(report)


# Simulate a benchmark run with pre-generated data
# (In practice this would run your actual retrieval + LLM)
simulated_results = pd.DataFrame([
    {'question': 'How does attention work?', 'P@3': 0.67, 'R@5': 0.80, 'MRR': 0.83,
     'ragas_score': 0.74, 'faithfulness': 0.85, 'answer_relevance': 0.80, 'context_precision': 0.67},
    {'question': 'What is gradient descent?', 'P@3': 1.0, 'R@5': 1.0, 'MRR': 1.0,
     'ragas_score': 0.88, 'faithfulness': 0.90, 'answer_relevance': 0.92, 'context_precision': 0.90},
    {'question': 'Explain KV cache', 'P@3': 0.33, 'R@5': 0.50, 'MRR': 0.50,
     'ragas_score': 0.55, 'faithfulness': 0.60, 'answer_relevance': 0.70, 'context_precision': 0.40},
    {'question': 'What is LoRA?', 'P@3': 0.67, 'R@5': 0.67, 'MRR': 1.0,
     'ragas_score': 0.71, 'faithfulness': 0.75, 'answer_relevance': 0.78, 'context_precision': 0.67},
    {'question': 'Describe RLHF pipeline', 'P@3': 0.33, 'R@5': 0.67, 'MRR': 0.33,
     'ragas_score': 0.45, 'faithfulness': 0.40, 'answer_relevance': 0.55, 'context_precision': 0.33},
])

# Plot the results heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Heatmap
ax = axes[0]
metric_cols = ['P@3', 'R@5', 'MRR', 'faithfulness', 'answer_relevance', 'context_precision', 'ragas_score']
heat_data = simulated_results[metric_cols].values
im = ax.imshow(heat_data, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(metric_cols)))
ax.set_xticklabels([m.replace('_', '\n') for m in metric_cols], fontsize=9)
ax.set_yticks(range(len(simulated_results)))
ax.set_yticklabels([q[:25] for q in simulated_results['question']], fontsize=9)
for i in range(len(simulated_results)):
    for j in range(len(metric_cols)):
        val = heat_data[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8,
                color='white' if val < 0.5 else 'black')
plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
ax.set_title('RAG Pipeline Metrics Heatmap', fontsize=12, fontweight='bold')

# Mean scores bar chart
ax = axes[1]
means = simulated_results[metric_cols].mean()
bar_colors = ['#2196F3' if m in ['P@3', 'R@5', 'MRR'] else
              '#4CAF50' if m in ['faithfulness', 'answer_relevance'] else
              '#FF9800' for m in metric_cols]
bars = ax.bar(range(len(metric_cols)), means.values, color=bar_colors, alpha=0.85)
ax.bar_label(bars, fmt='{:.2f}', padding=2, fontsize=9)
ax.set_xticks(range(len(metric_cols)))
ax.set_xticklabels([m.replace('_', '\n') for m in metric_cols], fontsize=9)
ax.axhline(0.7, color='gray', linestyle='--', alpha=0.6, label='Target ≥ 0.7')
ax.set_ylim(0, 1.1)
ax.set_ylabel('Mean Score')
ax.set_title('Average Scores Across Test Set', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3, axis='y')

from matplotlib.patches import Patch
legend_el = [Patch(facecolor='#2196F3', label='Retrieval'),
             Patch(facecolor='#4CAF50', label='Generation'),
             Patch(facecolor='#FF9800', label='RAGAS')]
ax.legend(handles=legend_el, loc='lower right')

plt.tight_layout()
plt.savefig('../figures/06_rag_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

# Diagnostic report
print(simulated_results[metric_cols].mean().to_string())

## 5. Engineering Notes

### Which Metrics to Use When

| Goal | Use These Metrics |
|------|------------------|
| Measure retrieval quality | P@K, R@K, NDCG@K, MRR |
| Detect hallucinations | Faithfulness |
| Check answer usefulness | Answer relevance |
| Diagnose noisy retrieval | Context precision |
| Measure end-to-end | RAGAS score |
| Production monitoring | RAGAS + LLM judge on samples |

### Production RAG Evaluation Stack

```python
# Recommended evaluation stack for production RAG

# 1. Offline evaluation (before deployment)
#    - Build a golden QA dataset: 200-500 questions with ground-truth answers
#    - Run RAGAS on full pipeline
#    - Compare P@K on retrieval vs manually labelled relevant docs

# 2. Online evaluation (in production)
#    - Log all queries + contexts + answers
#    - Run RAGAS faithfulness check on 10% sample (cost control)
#    - Track mean answer relevance over time (regression detection)
#    - Use thumbs up/down for human feedback collection

# 3. Tools
#    - ragas (pip install ragas) — full framework
#    - deepeval (pip install deepeval) — alternative
#    - trulens-eval — trace-level evaluation
#    - LangSmith — production tracing + evaluation
```

### The Evaluation Flywheel

```
Collect QA pairs → Evaluate → Find failures → Fix pipeline → Re-evaluate
      ↑                                                          |
      └──────────────── continuous improvement ─────────────────┘
```

## 6. Exercises

1. **Golden Dataset**: Build a 50-question evaluation set for a domain of your choice (e.g., a company FAQ). Label relevant doc IDs for each question.

2. **Chunking vs Retrieval**: Use the golden dataset to compare chunking strategies (fixed-size, sentence, semantic) on P@3 and R@5. Which strategy wins?

3. **Faithfulness Deep Dive**: Take 20 RAG answers and manually label each sentence as "supported" or "unsupported" by the context. Compare your labels vs the automated faithfulness metric. Where does the metric fail?

4. **RAGAS Integration**: Install the official `ragas` library and compare its faithfulness scores to our custom implementation on the same test cases.

5. **A/B Testing**: Build two RAG variants (different embedding model OR different chunk size). Run both on your golden dataset. Is the performance difference statistically significant (use paired t-test)?

---

## 7. References

- [Es et al. (2023) — RAGAS: Automated Evaluation of Retrieval Augmented Generation](https://arxiv.org/abs/2309.15217)
- [Saad-Falcon et al. (2023) — ARES: An Automated Evaluation Framework for Retrieval-Augmented Generation Systems](https://arxiv.org/abs/2311.09476)
- [Chen et al. (2024) — CRUD-RAG: A Comprehensive Chinese Benchmark for RAG Systems](https://arxiv.org/abs/2401.17043)
- [RAGAS Documentation](https://docs.ragas.io/)
- [Retrieval Evaluation in IR — Manning et al., Introduction to Information Retrieval (Ch. 8)](https://nlp.stanford.edu/IR-book/)